# AI Call Features (By Trial)

按**试次**分类，每个试次为一个数据点，分析AI调用特征。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 输入文件路径
data_dir = Path('../../../data')
ai_freq_file = data_dir / 'analysis/ai_freq/ai_freq_metrics.csv'
pre_call_file = data_dir / 'analysis/pre_call/pre_call_metrics.csv'
post_call_file = data_dir / 'analysis/post_call/post_call_metrics.csv'
ai_events_file = data_dir / 'pickle/ai_events.pkl'
user_msgs_file = data_dir / 'pickle/user_msgs.pkl'
participants_file = data_dir / 'pickle/participants.pkl'

# 输出文件路径
output_dir = Path('output')
ai_call_features_by_trial_file = output_dir / 'ai_call_features_by_trial.csv'
output_dir.mkdir(exist_ok=True)

In [ ]:
# 加载数据
ai_freq = pd.read_csv(ai_freq_file)
ai_events = pd.read_pickle(ai_events_file)
user_msgs = pd.read_pickle(user_msgs_file)
participants = pd.read_pickle(participants_file)
pre_call = pd.read_csv(pre_call_file)
post_call = pd.read_csv(post_call_file)

## 聚合数据

In [ ]:
# 先按被试+trial 聚合，确保每个数据点对应一个具体试次
trial_stats = []

group_cols = ['participant_id', 'trial_index']
for (pid, trial_idx), grp in ai_freq.groupby(group_cols):
    # 该试次总调用次数
    trial_calls = grp['ai_call_count'].sum()

    # 首次调用时间：仅在该试次存在调用时统计
    first_times_valid = grp['first_ai_time'].dropna()
    first_ai_time = first_times_valid.mean() if len(first_times_valid) > 0 else np.nan

    # 末次调用时间：仅在该试次存在调用时统计
    last_times_valid = grp['last_ai_time'].dropna()
    last_ai_time = last_times_valid.mean() if len(last_times_valid) > 0 else np.nan

    # 首尾调用间密度
    density_valid = grp['density_first_last'].dropna()
    density_first_last = density_valid.mean() if len(density_valid) > 0 else np.nan

    # 首次调用前想法数：仅在该试次存在调用时统计
    pre_ideas_valid = grp['pre_first_call_ideas'].dropna()
    pre_first_call_ideas = pre_ideas_valid.mean() if len(pre_ideas_valid) > 0 else np.nan

    # 调用间隔标准差：仅取调用次数>=3的记录
    multi_call_data = grp[grp['ai_call_count'] >= 3]
    interval_std_valid = multi_call_data['ai_interval_std'].dropna()
    ai_interval_std = interval_std_valid.mean() if len(interval_std_valid) > 0 else np.nan

    # 三个时间阶段调用次数
    stage1_count = grp['stage1_count'].sum()
    stage2_count = grp['stage2_count'].sum()
    stage3_count = grp['stage3_count'].sum()

    has_called = (trial_calls > 0)

    trial_stats.append({
        'participant_id': pid,
        'trial_index': trial_idx,
        'trial_calls': trial_calls,
        'first_ai_time': first_ai_time,
        'last_ai_time': last_ai_time,
        'density_first_last': density_first_last,
        'pre_first_call_ideas': pre_first_call_ideas,
        'ai_interval_std': ai_interval_std,
        'stage1_count': stage1_count,
        'stage2_count': stage2_count,
        'stage3_count': stage3_count,
        'has_called': has_called
    })

trial_df = pd.DataFrame(trial_stats)
print('试次级行为数据形状:', trial_df.shape)
trial_df = trial_df.merge(participants, on='participant_id', how='left')
print('合并特质后的试次级数据形状:', trial_df.shape)

# 合并 pre/post（调用级）
call_combined = pre_call.merge(
    post_call,
    on=['participant_id', 'item_name', 'trial_index', 'click_index'],
    how='inner'
)

# 聚合到被试+trial
call_trial = call_combined.groupby(['participant_id', 'trial_index']).agg({
    'pre_think_time': 'mean',
    'after_ai_time': 'mean',
    'relative_pause': 'mean',
    'perspective_taking': 'mean',
    'affected_by_ai': 'mean',
    'mean_ai_distance': 'mean'
}).reset_index()

# 转换为百分比（×100）
call_trial['perspective_taking'] = call_trial['perspective_taking'] * 100
call_trial['affected_by_ai'] = call_trial['affected_by_ai'] * 100

# 合并到试次级特征表
trial_df = trial_df.merge(call_trial, on=['participant_id', 'trial_index'], how='left')
print('合并调用级特征后的试次级数据形状:', trial_df.shape)

trial_df.to_csv(ai_call_features_by_trial_file, index=False, encoding='utf-8-sig')

## 数据总览

各个字段的均值、标准差、最小值、最大值等统计信息。

In [ ]:
metrics_cols = ['trial_calls', 'first_ai_time', 'pre_first_call_ideas', 'pre_think_time', 'perspective_taking', 'affected_by_ai']
stats = trial_df[metrics_cols].describe(percentiles=[])
stats.loc['median'] = trial_df[metrics_cols].median()
stats = stats.reindex(['count', 'mean', 'std', 'median', 'min', 'max'])
stats

## 调用次数分布图

In [ ]:
# 每个 trial 的调用次数分布
trial_calls = trial_df['trial_calls'].dropna()
print("试次调用次数描述统计：")
print(trial_calls.describe())

plt.figure(figsize=(10, 3))
sns.boxplot(x=trial_calls)
plt.xlabel('单个试次的AI调用次数')
plt.title('试次调用次数箱型图')
plt.show()

## 首次调用时间分布

In [ ]:
first_ai = trial_df['first_ai_time'].dropna()
print("试次首次调用时间描述统计：")
print(first_ai.describe())

plt.figure(figsize=(10,6))
sns.histplot(first_ai, bins=20, kde=True)
plt.xlabel('首次调用时间（秒）')
plt.ylabel('试次数')
plt.title('试次首次调用时间分布')
plt.show()

## 调用间隔稳定性分布

In [ ]:
interval_std = trial_df['ai_interval_std'].dropna()
print("试次调用间隔标准差描述统计（基于有>=3次调用的记录）：")
print(interval_std.describe())

plt.figure(figsize=(10,6))
sns.histplot(interval_std, bins=20, kde=True)
plt.xlabel('调用间隔标准差（秒）')
plt.ylabel('试次数')
plt.title('试次调用间隔稳定性分布')
plt.show()

## 调用前想法数量分布

In [ ]:
pre_ideas = trial_df['pre_first_call_ideas'].dropna()
print("试次首次调用前想法数描述统计：")
print(pre_ideas.describe())

plt.figure(figsize=(10,6))
sns.histplot(pre_ideas, bins=range(0, int(pre_ideas.max())+2), discrete=False)
plt.xlabel('首次调用前想法数')
plt.ylabel('试次数')
plt.title('试次首次调用前想法数分布')
plt.show()

## 零调用者比例

In [ ]:
zero_call_trials = (trial_df['has_called'] == False).sum()
print(f"总试次数: {len(trial_df)}")
print(f"零调用试次数: {zero_call_trials}")
print(f"零调用试次比例: {zero_call_trials/len(trial_df):.2%}")

## 阶段分布

In [ ]:
stage_cols = ['stage1_count', 'stage2_count', 'stage3_count']
stage_means = trial_df[stage_cols].mean()
stage_sems = trial_df[stage_cols].sem()

plt.figure(figsize=(8,6))
x = ['0-100秒', '100-200秒', '200-300秒']
plt.bar(x, stage_means, yerr=stage_sems, capsize=5)
plt.xlabel('时间段')
plt.ylabel('平均调用次数（单个试次）')
plt.title('试次在各阶段的调用次数均值（±标准误）')
plt.show()

## 调用前和调用后指标

In [ ]:
# 输出采纳比例（试次级均值）
print("直接采纳比例：", trial_df['perspective_taking'].mean())
print("间接采纳比例：", trial_df['affected_by_ai'].mean())

In [ ]:
import json
from pathlib import Path
Path('output').mkdir(exist_ok=True)
m = {}
for col in ['ai_call_count','first_ai_time','pre_first_call_ideas','ai_interval_std']:
    s = ai_freq[col].dropna()
    m[col] = {"M": round(float(s.mean()),4), "SD": round(float(s.std()),4),
              "Mdn": round(float(s.median()),4), "Min": int(s.min()),
              "Max": int(s.max()), "N": int(len(s))}
r = {
    "metrics": m,
    "total_trials": int(len(ai_freq)),
    "zero_trials": int((ai_freq['ai_call_count']==0).sum()),
    "adoption_perspective": round(float(post_call['perspective_taking'].mean()),6),
    "adoption_affected": round(float(post_call['affected_by_ai'].mean()),6),
    "stage_means": {c: round(float(ai_freq[c].mean()),4) for c in ['stage1_count','stage2_count','stage3_count']},
    "stage_sems": {c: round(float(ai_freq[c].sem()),4) for c in ['stage1_count','stage2_count','stage3_count']}
}
Path('output/trial_descriptives.json').write_text(json.dumps(r, ensure_ascii=False, indent=2))
print("Exported trial_descriptives.json")